## Introduction

This notebook ingests WDL tasks from the WILDS WDL Library into a [ChromaDB](https://docs.trychroma.com/) vector store for use in retrieval-augmented generation (RAG).

We store entire WDL tasks as 'documents' and use their `meta` sections to populate the document metadata. We plan to retrieve tasks using keyword filtering on this metadata.

**Keyword filtering strategy details:**
Each WDL task in the library has custom metadata tags describing what the tool does and its file inputs/outputs. We extract only relevant metadata that would be used for keyword filtering. We will derive which keywords to use for filtering based on selections user make from pre-defined inputs.

**Ingestion example:**

Starting task`meta` tags:

```{python}
# From 'strelka_germline' task in ww-strelka.wdl

    topic: "genomics,dna_polymorphism"
    species: "eukaryote"
    operation: "variant_calling"
    input_sample_required: "bam:nucleic_acid_sequence_alignment:bam,bai:data_index:bai"
    input_sample_optional: "target_regions_bed:annotation_track:bed"
    input_reference_required: "ref_fasta:dna_sequence:fasta,ref_fasta_index:data_index:fai"
    input_reference_optional: "none"
    output_sample: "variants_vcf:sequence_variations:vcf,variants_vcf_index:data_index:tbi"
    output_reference: "none"
```

Final ChromaDB document metadata:

```{python}
collection.add(
    ids=["id1"],
    documents=["entire wdl task"],
    metadatas=[{
        "tool": "strelka",
        "task": "strelka_germline",
        "topic": ["genomics", "dna_polymorphism"],
        "species": ["eukaryote"],
        "operation": "variant_calling",
        "input_sample_data_types": ["nucleic_acid_sequence_alignment", "data_index"],
        "input_sample_format_types": ["bam", "bai"],
        "output_sample_data_types": ["sequence_variations", "data_index"]
    }],
)
```

## 1. Imports and WDL loading

Copy module WDLs from a clone of [get-wilds/wilds-wdl-library](https://github.com/getwilds/wilds-wdl-library) before running this notebook, e.g.:

`cp ~/workspace/wilds-wdl-library/modules/*/ww-*.wdl ~/workspace/wilds-wdl-writer/data/wdl/`

In [1]:
import chromadb
from ingestion import load_wdl_tasks, update_collection

## 2. Extract WDL tasks and parse their metadata

In [2]:
wdl_dir = '../data/wdl/'

metas_tasks = load_wdl_tasks(wdl_dir)

In [3]:
metas_tasks[:3]

[({'tool': 'cnvkit',
   'task': 'create_reference',
   'topic': ['genomics', 'copy_number_variation'],
   'species': ['human', 'eukaryote'],
   'operation': ['indexing'],
   'input_sample_data_types': ['nucleic_acid_sequence_alignment',
    'data_index'],
   'input_sample_format_types': ['bam', 'bai'],
   'output_sample_data_types': ['none']},
  'task create_reference {\n  meta {\n    author: "Taylor Firman"\n    email: "tfirman@fredhutch.org"\n    description: "Create CNVkit reference from normal samples or pooled reference"\n    url: "https://raw.githubusercontent.com/getwilds/wilds-wdl-library/refs/heads/main/modules/ww-cnvkit/ww-cnvkit.wdl"\n    outputs: {\n        reference_cnn: "CNVkit reference file (.cnn)"\n    }\n    topic: "genomics,copy_number_variation"\n    species: "human,eukaryote"\n    operation: "indexing"\n    input_sample_required: "bam_files:nucleic_acid_sequence_alignment:bam,bam_indices:data_index:bai"\n    input_sample_optional: "target_bed:annotation_track:bed,a

## 3. Do some quick QC

Look at the unique tags we have for each category, to do a spot-check that things look ok.

In [4]:
def collect_unique_values(tuples_list):
    unique_values = {}
    for meta, *_ in tuples_list:
        for key, values in meta.items():
            if key in ('tool', 'task'):
                continue
            if key not in unique_values:
                unique_values[key] = set()
            unique_values[key].update(values)
    return {k: sorted(v) for k, v in unique_values.items()}

unique_terms = collect_unique_values(metas_tasks)

for key, values in unique_terms.items():
    print(key)
    print(values)

topic
['any', 'copy_number_variation', 'data_quality_management', 'dna_mutation', 'dna_packaging', 'dna_polymorphism', 'epigenomics', 'gene_expression', 'genomics', 'mapping', 'metagenomics', 'nucleic_acid_structure_analysis', 'protein_disordered_structure', 'protein_expression', 'protein_structure_analysis', 'proteomics', 'public_health_and_epidemiology', 'ribosome_profiling', 'rna_splicing', 'sequence_assembly', 'sequence_features', 'sequencing', 'structural_variation', 'transcriptomics']
species
['eukaryote', 'human', 'prokaryote', 'virus']
operation
['aggregation', 'alternative_splicing_prediction', 'annotation', 'copy_number_variation_detection', 'data_deposition', 'data_filtering', 'data_formatting', 'data_handling', 'data_retrieval', 'file_handling', 'indel_detection', 'indexing', 'mapping', 'nucleic_acid_structure_analysis', 'protein_structure_prediction', 'quality_control', 'quantification', 'rna_secondary_structure_prediction', 'rna_seq_quantification', 'sequence_alignment', 

## 4. Make (or load) ChromaDB collection and add WDL tasks + metadata

In [5]:
collection_count = update_collection(metas_tasks)

Do some minor QC: We expect the number of items in the collection to be the same as the number of WDL tasks.

In [6]:
num_tasks = len(metas_tasks)
print(f"WDL tasks ingested: {num_tasks}")
print(f"Tasks in collection: {collection_count}")
print(f"Same number of tasks ingested and in collection: {num_tasks == collection_count}")

WDL tasks ingested: 119
Tasks in collection: 119
Same number of tasks ingested and in collection: True


Recall that we are not entering ALL the tasks from the WILDS WDL Library (we are skipping the `ww-testdata.wdl` and `ww-template.wdl`)

## 5. Quickly confirm we can filter by metadata

In [7]:
def get_collection(chroma_dir, collection_name="wdl_tasks"):
    client = chromadb.PersistentClient(path=chroma_dir)
    return client.get_collection(name=collection_name)

collection = get_collection('../data/chroma/')

### Define terms we want to filter on

In this example, we only want tasks that take aligned, indexed data as input and call variants. In the future these terms would come from interpreting pre-defined user inputs.

In [8]:
my_input_data = ["nucleic_acid_sequence_alignment", "data_index"]
my_input_format = ["bam", "bai"]
my_output_data = ["sequence_variations"]

### Write the filter

ChromaDB wants a nested list of dictionaries containing metadata keys to filter on, filter logic (e.g. `$contains`, `$in`) and the keywords terms themselves. See their [metadata filtering](https://docs.trychroma.com/docs/querying-collections/metadata-filtering) docs page.

In [9]:
# Generate filters (will use 'and' to ensure task meets all minimum needs)
filter_input_data = [{"input_sample_data_types": {"$contains": i}} for i in my_input_data]
filter_input_format = [{"input_sample_format_types": {"$contains": i}} for i in my_input_format]
filter_output_data = [{"output_sample_data_types": {"$contains": i}} for i in my_output_data]

# Look at what we'll be filtering on
print(filter_input_data)
print(filter_input_format)
print(filter_output_data)

# Combine
full_filter = filter_input_data + filter_input_format + filter_output_data

[{'input_sample_data_types': {'$contains': 'nucleic_acid_sequence_alignment'}}, {'input_sample_data_types': {'$contains': 'data_index'}}]
[{'input_sample_format_types': {'$contains': 'bam'}}, {'input_sample_format_types': {'$contains': 'bai'}}]
[{'output_sample_data_types': {'$contains': 'sequence_variations'}}]


### Retrieve 'documents' (WDL tasks) passing the filter

Below are printed the the ids of documents that passed the filter and are retrieved from our collection. Recall that each 'id' is: `<tool name>_<task name>`

In [10]:
retrieved = collection.get(where={"$and": full_filter})

retrieved['ids']

['cnvkit_run_cnvkit',
 'delly_delly_call',
 'bcftools_mpileup_call',
 'deepvariant_run_deepvariant',
 'smoove_smoove_call',
 'manta_manta_call',
 'glimpse2_glimpse2_phase_cram',
 'strelka_strelka_germline',
 'strelka_strelka_somatic',
 'gatk_haplotype_caller',
 'gatk_mutect2',
 'gatk_haplotype_caller_parallel',
 'gatk_mutect2_parallel',
 'clair3_run_clair3']

## Notes for next steps

- Multiple categories contain `"any"`, and those should be included in our filtering logic

- We can use `llamaindex` methods to retrieve documents from the chromadb document store, e.g.:
    - `from llama_index.vector_stores.chroma import ChromaVectorStore`
    - `from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator`
    - `from llama_index.core import VectorStoreIndex`